# Day 21 — Checkpoint Solution (Exemplar)

Synthetic-mode run with the planted premium; the real-mode code path is
identical (flip QRC_DATA; expected-pattern readings annotated).

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_multi(X_cols, y):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in X_cols])
    y = np.asarray(y, float); n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y; e = y - X @ b
    s2 = e @ e / (n - k)
    r2 = 1 - (e@e)/((y-y.mean())@(y-y.mean()))
    return {"b": b, "se": np.sqrt(np.diag(s2*XtX_inv)), "r2": r2}

def fama_macbeth(frame, ycol, xcols):
    rows, r2s = {}, {}
    for m, g in frame.groupby(level=0):
        g = g[[ycol] + xcols].dropna()
        if len(g) < 10: continue
        Zs = [(g[c]-g[c].mean())/g[c].std() for c in xcols]
        f = ols_multi(Zs, g[ycol].values)
        rows[m] = f["b"]; r2s[m] = f["r2"]
    lam = pd.DataFrame(rows, index=["const"] + xcols).T
    T = len(lam)
    out = pd.DataFrame({"mean": lam.mean(), "SE": lam.std(ddof=1)/np.sqrt(T),
                        "t": lam.mean()/(lam.std(ddof=1)/np.sqrt(T))})
    return lam, out, float(pd.Series(r2s).mean())

## A0 — the pre-declaration cell (written before any results)

```
UNIVERSE: 50 names (synthetic world, or research50 in real mode). WINDOW: full sample.
CHARACTERISTIC (single, pre-declared): mom_12-2 = cumulative log return, months t-12..t-2.
RESPONSE: month t+1 log return. ENGINE: Fama-MacBeth monthly cross-sections,
SE = sd(lambda_t)/sqrt(T); slope ACF displayed; NW(6) if |rho1| > 0.15.
CONTROLS: null characteristic (frozen random z) must sit |t| < 2.5; planted
+40bp/sig premium must be recovered within ~2 SEs (synthetic mode).
SKIP JUSTIFICATION (referee-quotable): "The most recent month is skipped to
exclude short-term reversal, which is a distinct, opposite-signed phenomenon
(Jegadeesh & Titman 1993); including it would contaminate the premium we test."
```

## A1 — the build + audit

In [ ]:
if DATA_SOURCE == "real":
    from qrc.data import get_prices
    from qrc.universe import load_universe
    names = load_universe("research50")[:50]
    px = get_prices(names, start="2010-01-01")
    rets_d = np.log(px).diff()
else:
    from qrc.synth import synthetic_returns
    names = [f"S{i}" for i in range(50)]
    rets_d = synthetic_returns(n_days=3600, n_assets=50, seed=8, corr=0.4,
                               drift_spread=0.0002)
    rets_d.columns = names

per = rets_d.index.to_period("M")
r_m = rets_d.groupby(per).sum(); r_m.index = r_m.index.to_timestamp()
mom_m = r_m.rolling(11).sum().shift(1)          # months t-12..t-2, known by end of month t

if DATA_SOURCE == "synthetic":                   # the planted positive control
    zmom = mom_m.sub(mom_m.mean(axis=1), axis=0).div(mom_m.std(axis=1), axis=0)
    r_m_planted = r_m + 0.0040 * zmom.fillna(0.0)
else:
    r_m_planted = r_m

frame = pd.DataFrame({"mom": mom_m.stack()})
frame["r_next"] = r_m_planted.stack().groupby(level=1).shift(-1)
frame["znull"] = np.tile(np.random.default_rng(11).normal(0, 1, len(names)), len(r_m))
frame = frame.dropna(subset=["r_next"]); frame.index.names = ["month", "name"]

t0 = frame.index.get_level_values(0)[200]
last_char_day = rets_d.index[rets_d.index.to_period("M") <= t0.to_period("M")][-1]
first_resp_day = rets_d.index[rets_d.index.to_period("M")
                              == (t0 + pd.offsets.MonthBegin(1)).to_period("M")][0]
print(f"rows {len(frame)} | months {frame.index.get_level_values(0).nunique()}")
print(f"AUDIT: mom inputs end {last_char_day.date()}, response starts {first_resp_day.date()}: {last_char_day < first_resp_day}")

**Reading.** Rubric 1–3 artifacts: the pre-declaration cell, the build,
the audit line printing True. Non-overlapped monthly design
acknowledged (largely NW-free by construction — no forward-sum
overlap).

## A2 — controls

In [ ]:
_, null_t, _ = fama_macbeth(frame.dropna(subset=["znull"]), "r_next", ["znull"])
_, mom_t, _ = fama_macbeth(frame.dropna(subset=["mom"]), "r_next", ["mom"])
n_ok = abs(null_t.loc["znull", "t"]) < 2.5
if DATA_SOURCE == "synthetic":
    gap = (mom_t.loc["mom", "mean"] - 0.0040) / mom_t.loc["mom", "SE"]
    print(f"NULL: t {null_t.loc['znull','t']:+.2f} -> {'PASS' if n_ok else 'FAIL'} | "
          f"POSITIVE: {mom_t.loc['mom','mean']*1e4:+.0f}bp/sig, {gap:+.1f} SEs from +40bp -> {'PASS' if abs(gap)<2 else 'FAIL'}")
else:
    print(f"NULL: t {null_t.loc['znull','t']:+.2f} -> {'PASS' if n_ok else 'FAIL'} "
          f"(positive control was exercised in synthetic mode before this run was believed)")

**Expected (synthetic control world).** NULL t within ±1ish (PASS);
positive control: λ̂_mom ≈ +33–45bp/σ — note the plant's feedback (mom
computed on raw returns; planted component enters later mom's, diluting
slightly) — recovery within ~1.5 SE of +40bp ⇒ PASS.

## A3 — both readings

In [ ]:
def quintile_report(frame, char, ycol="r_next"):
    rows = {}
    for m, g in frame.groupby(level=0):
        g = g[[char, ycol]].dropna()
        if len(g) < 25: continue
        q = pd.qcut(g[char].rank(method="first"), 5, labels=False)
        rows[m] = g.groupby(q)[ycol].mean()
    Q = pd.DataFrame(rows).T.dropna(); T = len(Q)
    spread = Q[4] - Q[0]
    return Q.mean(), spread.mean(), spread.mean()/(spread.std(ddof=1)/np.sqrt(T))

q_means, sp, sp_t = quintile_report(frame, "mom")
print("sorts bp:", " ".join(f"{q_means[i]*1e4:+6.0f}" for i in range(5)),
      f"| spread {sp*1e4:+.0f}bp t={sp_t:+.1f}")
lam, out, r2m = fama_macbeth(frame.dropna(subset=["mom"]), "r_next", ["mom"])
print(f"FM: lambda {out.loc['mom','mean']*1e4:+.1f} bp/sig  t {out.loc['mom','t']:+.1f}  R2bar {r2m:.3f}")

**One story, two readings (synthetic).** Monotone-ish quintile ladder,
spread ≈ +90bp t ≈ 4–6, FM λ̂ ≈ +35–45bp/σ t ≈ 4–6.
Real mode (expected pattern): the ladder is noisier — megacap
momentum runs t ≈ 1–2 on λ̂ with the spread reading usually larger in
magnitude than the slope (deciles vs continuous line — the decile
reading hides the middle fog) — and the REPORT shows both, aligned.

## A4 — certified SEs + the regime test

In [ ]:
lamser = lam["mom"]
rho1 = lamser.autocorr(1)
print(f"lambda_t ACF lag-1 {rho1:+.2f} -> {'plain sd/sqrt(T) stands' if abs(rho1)<0.15 else 'NW applied'}")

half = len(lamser)//2
d1, d2 = lamser.iloc[:half].mean(), lamser.iloc[half:].mean()
se_d = np.sqrt(lamser.iloc[:half].var(ddof=1)/half + lamser.iloc[half:].var(ddof=1)/(len(lamser)-half))
band_roll_36 = 2 * lamser.std(ddof=1) / np.sqrt(36)
print(f"regime arithmetic: 1st half {d1*1e4:+.0f}bp, 2nd half {d2*1e4:+.0f}bp, "
      f"Delta {(d2-d1)*1e4:+.0f}bp, t_Delta {(d2-d1)/se_d:+.1f}, rolling-36 band ±{band_roll_36*1e4:.0f}bp")

**Reading.** Rubric 6: ACF displays ≈ 0 (independent monthly
cross-sections; if |ρ1| > 0.15 the NW(6) row was mandatory — in this
world it isn't). Rubric 7: the half-split Δ is printed WITH its t and
the rolling-band scale; unless |Δ|/band is decisively >1, the report's
regime section says *"no regime claim made — halves differ within the
band"*. (Real mode: expect Δ between -50 and +100bp with band ~±60–90bp
— usually within, occasionally just outside 2008-09's corner, which is
flagged not claimed.)

## Exercise B — the report (exemplar, numbers from the actual run)

```
TITLE: The 12-2 momentum premium in a 50-name synthetic-control panel, 3600d (~14y)

1. QUESTION & DESIGN. Does 12-2 momentum (cumulative log return months
   t-12..t-2) forecast month t+1 returns cross-sectionally? Pre-declared
   single characteristic, monthly non-overlapping response, FM engine;
   the most recent month is skipped to exclude short-term reversal, a
   distinct opposite-signed phenomenon (JT93).

2. DATA & CONSTRUCTION. 50-name synthetic control world (corr 0.4 one-
   factor structure); audit line True (mom inputs end month t-1's close;
   response begins month t+1's first bar). No composites, no accounting
   proxies, no survivorship by construction (synthetic) / flagged
   (real: research50's recovered-universe bounds the bias at module-05
   numbers, direction positive).

3. CONTROLS. Null t = <number> PASS; positive control recovered
   <number> bp/sig vs +40bp planted (<number> SEs) PASS. Machinery
   certified before results were read.

4. RESULTS. Sorts: Q1..Q5 = <numbers> bp, spread <number>bp, t <number>.
   FM: lambda = <number> bp/sig, t <number>, R-bar2 <number> (floor
   1/49 = 0.02), ACF1 <number>. Regime: half-split Delta <number>bp,
   t_Delta <number>, rolling-36 band +/-<number>bp -> <verdict line>.
   Both readings agree.

5. CAVEATS. proxy: none (price-only characteristic, low) | survivorship:
   synthetic none / real bounded +1-2%/yr (medium for level claims) |
   truncation: megacap dispersion compressed (real; medium) |
   construction: skip enforced (solved) | multiplicity: one char,
   two readings (low) | slope ACF: displayed, within tolerance (low).

6. VERDICT. In-universe, 12-2 momentum earns a detectable premium at
   ~+40bp/sig monthly (THIS synthetic control world; real-mode reading:
   suggestive t ~1-2, regime-pooled, power-limited at 50 names) with
   certified controls and certified SEs; no regime claim survives the
   band. NEXT EXPERIMENT: module 07's full JT93 — 1000-name universe,
   formation/holding grid, factor model evaluation of the long-short.

SELF-SCORE: 10/10 lines held (synthetic exemplar).
```

**Where candidates actually lose rubric lines.** Rubric 4 (controls
skipped "because real mode"), rubric 6 (a pooled-classical t sneaks
into the sorts' SE via std over observations instead of months), rubric
7 (a regime adjective sneaks in without the band), rubric 9 (a prose
number rounded from memory instead of re-quoted from the cell). Every
one of those is a habit, not a knowledge gap — which is the entire
point of rehearsing the drill.